# Kaggle GPU RNN/LSTM Image Captioning preinject_v2

Notebook ini dibuat untuk Kaggle Notebook dengan accelerator **GPU T4 x2**.

Input yang dipakai ada 2:
1. **Project code dataset**: zip kecil berisi kode repo (`src/rnn`, notebook, `requirements.txt`). Tidak perlu upload gambar Flickr8k, `images_feature`, atau `artifacts`.
2. **Kaggle Flickr 8k Dataset**: tambahkan dataset publik `adityajn105/flickr8k` lewat **Add Input**. Notebook juga mendukung mirror yang punya `Images/` + `captions.txt`, misalnya `srinivasac/flickr8k-dataset`.

Notebook akan mencari dataset Flickr di `/kaggle/input`, membuat struktur `RNN_dataset/captions.txt` dan `RNN_dataset/Images` di `/kaggle/working`, lalu membuat `images_feature/*.npy` kalau belum ada.

Catatan: kalau `images_feature` belum ikut di project zip, feature extraction pertama kali butuh InceptionV3 ImageNet weights. Aktifkan **Internet On** di Kaggle jika weights belum cached.


In [ ]:
from pathlib import Path
import os
import shutil
import sys

PROJECT_NAME = "Tubes2ML_KicauMania"
REFRESH_WORKING_COPY = False

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")


def has_code_markers(path: Path) -> bool:
    return (path / "src" / "rnn").exists()


def find_code_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if has_code_markers(candidate):
            return candidate

    if KAGGLE_INPUT.exists():
        for dataset_root in sorted(KAGGLE_INPUT.iterdir()):
            if has_code_markers(dataset_root):
                return dataset_root
            for marker in dataset_root.rglob("src/rnn"):
                repo = marker.parents[1]
                if has_code_markers(repo):
                    return repo

    raise FileNotFoundError(
        "Project code tidak ditemukan. Upload zip kecil project yang berisi folder src/rnn sebagai Kaggle Dataset."
    )


source_root = find_code_root()
if str(source_root).startswith(str(KAGGLE_INPUT)):
    repo_root = KAGGLE_WORKING / PROJECT_NAME
    if repo_root.exists() and REFRESH_WORKING_COPY:
        shutil.rmtree(repo_root)
    if not repo_root.exists():
        shutil.copytree(
            source_root,
            repo_root,
            ignore=shutil.ignore_patterns(
                ".git",
                "__pycache__",
                ".ipynb_checkpoints",
                "RNN_dataset",
                "CNN_dataset",
                "images_feature",
                "artifacts",
            ),
        )
else:
    repo_root = source_root

REPO_ROOT = repo_root.resolve()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Source root: {source_root}")
print(f"Working repo: {REPO_ROOT}")
print(f"Python imports from: {sys.path[0]}")


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("Physical GPUs:", gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print(f"Memory growth already initialized for {gpu}: {exc}")

if len(gpus) >= 2:
    strategy = tf.distribute.MirroredStrategy()
else:
    strategy = tf.distribute.get_strategy()

print("Strategy:", type(strategy).__name__)
print("Replicas:", strategy.num_replicas_in_sync)
assert strategy.num_replicas_in_sync >= 1


In [ ]:
import nltk

for package in ("wordnet", "omw-1.4"):
    try:
        nltk.data.find(f"corpora/{package}")
    except LookupError:
        try:
            nltk.download(package, quiet=True)
        except Exception as exc:
            print(f"NLTK {package} tidak bisa di-download; METEOR akan diisi NaN jika resource tidak tersedia. Detail: {exc}")


In [ ]:
from src.rnn.paths import ARCH_TAG, RnnPaths

paths = RnnPaths.from_root(REPO_ROOT)
SEQ_LEN = 35
EPOCHS = 5
EVAL_LIMIT = 100
PER_REPLICA_IMAGE_BATCH = 32
BATCH_SIZE = PER_REPLICA_IMAGE_BATCH * strategy.num_replicas_in_sync
RUN_FULL_TRAINING = True
RETRAIN_EXISTING_WEIGHTS = False
USE_TF_DATASET = True
FORCE_FEATURE_EXTRACTION = False
FEATURE_EXTRACTION_BATCH_SIZE = 64

print("Architecture:", ARCH_TAG)
print("Image batch size:", BATCH_SIZE)
print("Training history:", paths.training_history_file)
print("Weights dir:", paths.weights_dir)


In [ ]:
from src.rnn.feature_extraction import extract_and_save_repo_features


def has_jpgs(path: Path) -> bool:
    return path.exists() and path.is_dir() and next(path.glob("*.jpg"), None) is not None


def find_flickr8k_input() -> tuple[Path, Path]:
    # Kalau dataset sudah tersedia di working copy, pakai itu.
    if paths.captions_file.exists() and has_jpgs(paths.image_dir):
        return paths.captions_file, paths.image_dir

    if not KAGGLE_INPUT.exists():
        raise FileNotFoundError("/kaggle/input tidak ada. Jalankan cell ini di Kaggle atau sediakan RNN_dataset lokal.")

    preferred_names = ("flickr8k", "flickr8k-dataset", "flickr-8k", "flickr_8k")
    roots = sorted(
        KAGGLE_INPUT.iterdir(),
        key=lambda p: (not any(name in p.name.lower() for name in preferred_names), p.name.lower()),
    )

    for root in roots:
        caption_files = list(root.rglob("captions.txt"))
        for caption_file in caption_files:
            parent = caption_file.parent
            image_candidates = [
                parent / "Images",
                parent / "images",
                parent / "Flickr8k_Dataset",
                parent / "Flicker8k_Dataset",
                root / "Images",
                root / "images",
                root / "Flickr8k_Dataset",
                root / "Flicker8k_Dataset",
            ]
            for image_dir in image_candidates:
                if has_jpgs(image_dir):
                    return caption_file, image_dir

            for image_dir in root.rglob("*"):
                if has_jpgs(image_dir):
                    return caption_file, image_dir

    raise FileNotFoundError(
        "Flickr8k Kaggle Dataset tidak ditemukan. Klik Add Input lalu tambahkan dataset `adityajn105/flickr8k`."
    )


def ensure_working_flickr_dataset():
    caption_src, image_src = find_flickr8k_input()
    paths.dataset_dir.mkdir(parents=True, exist_ok=True)

    if not paths.captions_file.exists() or paths.captions_file.resolve() != caption_src.resolve():
        shutil.copy2(caption_src, paths.captions_file)

    if not has_jpgs(paths.image_dir):
        if paths.image_dir.exists():
            shutil.rmtree(paths.image_dir)
        try:
            paths.image_dir.symlink_to(image_src, target_is_directory=True)
        except OSError:
            shutil.copytree(image_src, paths.image_dir)

    print("Flickr captions:", paths.captions_file)
    print("Flickr images:", paths.image_dir)


ensure_working_flickr_dataset()

features_ready = paths.features_file.exists() and paths.feature_names_file.exists()
if FORCE_FEATURE_EXTRACTION or not features_ready:
    print("Extracting InceptionV3 features. Jika gagal download weights, aktifkan Internet di Kaggle lalu rerun.")
    try:
        extract_and_save_repo_features(
            REPO_ROOT,
            batch_size=FEATURE_EXTRACTION_BATCH_SIZE,
            force=FORCE_FEATURE_EXTRACTION,
        )
    except Exception as exc:
        raise RuntimeError(
            "Feature extraction gagal. Pastikan Kaggle Internet On untuk download InceptionV3 ImageNet weights, "
            "atau upload images_feature/features.npy dan images_feature/image_names.npy di project code dataset."
        ) from exc
else:
    print("Feature cache found:", paths.feature_dir)

required_files = [paths.captions_file, paths.features_file, paths.feature_names_file]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("File wajib belum ada setelah setup data: " + ", ".join(missing))

print("Features:", paths.features_file)
print("Feature names:", paths.feature_names_file)


In [ ]:
from src.rnn.experiment import load_caption_sequences, load_text_util
from src.rnn.training import prepare_training_data

text_util = load_text_util(REPO_ROOT, sequence_length=SEQ_LEN)
caption_mapping, sequence_mapping = load_caption_sequences(REPO_ROOT, text_util)
training_data = prepare_training_data(REPO_ROOT, sequence_length=SEQ_LEN)
image_features_map = training_data["image_features"]

train_keys = set(training_data["train_keys"])
val_keys = set(training_data["val_keys"])
test_keys = set(training_data["test_keys"])
assert not (train_keys & val_keys)
assert not (train_keys & test_keys)
assert not (val_keys & test_keys)

train_stack = np.asarray([image_features_map[key] for key in training_data["train_keys"][:1000]], dtype=np.float32)
assert np.isfinite(train_stack).all()

print("Vocabulary size:", text_util.vocab_size)
print("Captions mapped:", len(caption_mapping))
print("Train/val/test:", len(train_keys), len(val_keys), len(test_keys))
print("Scaled feature sample mean/std:", float(train_stack.mean()), float(train_stack.std()))


In [ ]:
from src.rnn.modeling import build_caption_model, caption_steps
from src.rnn.utils.train_utils import make_caption_dataset

pad_id = text_util.word_to_idx.get("pad", 0)
smoke_dataset = make_caption_dataset(
    mapping_data=training_data["train_data"],
    image_features=image_features_map,
    sequence_length=text_util.sequence_length,
    batch_size=2,
    pad_id=pad_id,
)
(X_img, X_txt), y, sample_weight = next(iter(smoke_dataset))

assert X_txt.shape[1] == caption_steps(text_util.sequence_length)
assert y.shape[1] == caption_steps(text_util.sequence_length)
assert sample_weight.shape[1] == caption_steps(text_util.sequence_length)

with strategy.scope():
    smoke_model = build_caption_model(True, 1, 128, text_util.vocab_size, text_util.sequence_length)

assert smoke_model.output_shape[1] == caption_steps(text_util.sequence_length)
assert smoke_model.get_layer("PreInject_Concat").output.shape[1] == caption_steps(text_util.sequence_length) + 1
assert "Drop_Image_Timestep" in [layer.name for layer in smoke_model.layers]

loss = smoke_model.train_on_batch([X_img, X_txt], y, sample_weight=sample_weight)
print("Smoke batch:", X_img.shape, X_txt.shape, y.shape, sample_weight.shape)
print("Smoke train_on_batch loss:", float(loss) if not isinstance(loss, (list, tuple)) else loss)
del smoke_model


In [ ]:
from src.rnn.experiment import load_training_history
from src.rnn.training import train_all_variations, training_artifacts_complete

if RUN_FULL_TRAINING:
    history_df = train_all_variations(
        REPO_ROOT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        sequence_length=SEQ_LEN,
        retrain_existing=RETRAIN_EXISTING_WEIGHTS,
        strategy=strategy,
        use_tf_dataset=USE_TF_DATASET,
    )
elif training_artifacts_complete(REPO_ROOT):
    history_df = load_training_history(REPO_ROOT)
else:
    raise FileNotFoundError("History preinject_v2 belum ada. Set RUN_FULL_TRAINING=True untuk training di Kaggle GPU.")

display_columns = [
    "architecture", "model_type", "variation_name", "layers", "hidden_state",
    "final_loss", "final_val_loss", "training_time_sec", "weight_file", "training_status",
]
history_df[[column for column in display_columns if column in history_df.columns]]


In [ ]:
from src.rnn.ImageCaptioningScratch import ImageCaptioningModel
from src.rnn.experiment import make_keras_model, greedy_decode_keras

keras_model = make_keras_model(REPO_ROOT, text_util, "LSTM", "Shallow_Small", 1, 128)
scratch_model = ImageCaptioningModel(keras_model, text_util, is_lstm=True)

for image_name in training_data["test_keys"][:3]:
    keras_caption = greedy_decode_keras(keras_model, image_features_map[image_name], text_util, max_len=10)
    scratch_caption = scratch_model.generate_caption(image_features_map[image_name], max_len=10)
    print(image_name)
    print("keras  :", keras_caption)
    print("scratch:", scratch_caption)
    assert keras_caption == scratch_caption

del keras_model, scratch_model


In [ ]:
from src.rnn.experiment import (
    assert_anti_collapse_acceptance,
    compare_best_keras_vs_scratch,
    evaluate_all_variations,
    max_length_sweep,
    qualitative_samples,
    write_analysis_summary,
)

variation_results, caption_details = evaluate_all_variations(
    REPO_ROOT,
    split="test",
    limit=EVAL_LIMIT,
    max_len=SEQ_LEN,
)
display(variation_results.sort_values(["model_type", "scratch_bleu_4"], ascending=[True, False]))

keras_vs_scratch = compare_best_keras_vs_scratch(REPO_ROOT, split="test", limit=EVAL_LIMIT, max_len=SEQ_LEN)
display(keras_vs_scratch)

max_length_results = max_length_sweep(REPO_ROOT, lengths=(10, 20, 35), split="test", limit=EVAL_LIMIT)
display(max_length_results)

samples = qualitative_samples(REPO_ROOT, limit=EVAL_LIMIT, n_per_bucket=4)
display(samples)

acceptance = assert_anti_collapse_acceptance(REPO_ROOT)
display(acceptance[["model_type", "variation_name", "unique_captions", "top_caption_frequency"]])

summary_path = write_analysis_summary(REPO_ROOT)
print("Summary:", summary_path)


In [ ]:
from pathlib import Path
import zipfile

output_zip = Path("/kaggle/working/rnn_preinject_v2_outputs.zip") if Path("/kaggle/working").exists() else REPO_ROOT / "artifacts" / "rnn_preinject_v2_outputs.zip"
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [paths.results_dir, paths.weights_dir]:
        if folder.exists():
            for file in folder.rglob("*"):
                if file.is_file() and ("PreInjectV2" in file.name or file.suffix in {".csv", ".md"}):
                    zf.write(file, file.relative_to(REPO_ROOT))
    if paths.feature_scaler_file.exists():
        zf.write(paths.feature_scaler_file, paths.feature_scaler_file.relative_to(REPO_ROOT))

print("Output zip:", output_zip)
print("Done.")
